In [14]:
%load_ext autoreload
%autoreload 2

In [41]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

openai_client = OpenAI(
    api_key=os.getenv("GEMINI_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

In [42]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)
    
len(documents)

72

In [43]:
from minsearch import Index

index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(documents)

In [ ]:
index.search("How does the agentic loop keep calling the model until it stops?")

In [5]:
def build_index(documents):
    index = Index(
        text_fields=["content"],
        keyword_fields=["filename"]
    )

    index.fit(documents)
    return index

In [6]:
index = build_index(documents)

In [ ]:
print("My index is:", type(index))

In [ ]:
from rag_helper import RAGBase

assist = RAGBase(index, openai_client)
assist.rag("How does the agentic loop keep calling the model until it stops?")

In [45]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [26]:
len(chunks)

295

In [48]:
def building_index(chunks):
    index = Index(
        text_fields=["content"],
        keyword_fields=["filename"]
    )
    index.fit(chunks)
    return index

In [49]:
second_index = building_index(chunks)
print(type(second_index))

<class 'minsearch.minsearch.Index'>


In [ ]:
from rag_helper import RAGBase


assistant = RAGBase(second_index, openai_client)
assistant.rag("How does the agentic loop keep calling the model until it stops?")

In [5]:
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return second_index.search(
        query,
        num_results=5,
        boost_dict={"content": 3.0},
    )

In [31]:
instructions = '''
You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering.
'''

In [53]:
from pydantic_ai import Agent

agent = Agent(model='google:gemini-2.5-flash', system_prompt=instructions)

@agent.tool_plain
def search(query: str) -> dict:
    """Search the FAQ database for entries matching the given query."""
    return second_index.search(query, boost_dict={'content': 3.0}, num_results=5)

result = await agent.run("How does the agentic loop work, and how is it different from plain RAG?")
print(result.output)

An agentic loop is an iterative process where a Large Language Model (LLM), acting as an "agent," dynamically decides a sequence of steps to achieve a goal. This loop typically involves the LLM calling various tools, executing them, and then processing the results to inform its next action. This process continues, often with multiple turns and managing conversation history, until the agent determines it has reached a final answer or completed its objective. This allows for dynamic adaptation and decision-making by the LLM itself, rather than following a fixed, predetermined workflow.

Plain Retrieval Augmented Generation (RAG), on the other hand, is a technique focused on grounding an LLM's responses in specific data to prevent hallucinations. It typically involves a more linear, three-step pipeline:

1.  **Search/Retrieval:** Relevant information is retrieved from a data source (e.g., a knowledge base, documents).
2.  **Augmentation:** The retrieved information is then used to augment

In [ ]:
print(result.usage.requests)

3


/tmp/ipykernel_640/1523381592.py:3: PydanticAIDeprecationWarning: `AgentRunResult.usage` is no longer a method; access it as a property (drop the parentheses).
  print(result.usage().requests)
